# Scraping Komentar YouTube


In [ ]:
import pandas as pd
from dotenv import load_dotenv
import os
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

load_dotenv()
API_KEY = os.getenv("YOUTUBE_API_KEY")

if not API_KEY:
    raise ValueError("YOUTUBE_API_KEY tidak ditemukan di .env")

youtube = build("youtube", "v3", developerKey=API_KEY)


## Daftar Video


In [ ]:
VIDEO_IDS = [
    "OdNPdAJUxbU",
    "Ud0KZHj5MM8",
    "Q_RINy6niCg",
    "Y1LSxLRbS80",
]


## Fungsi Scraping


In [ ]:
def scrape_comments(video_id):
    """Ambil semua komentar top-level dari satu video."""
    comments = []

    request = youtube.commentThreads().list(
        part="snippet",
        videoId=video_id,
        maxResults=100,
    )

    while request:
        response = request.execute()

        for item in response["items"]:
            thread = item["snippet"]
            comment = thread["topLevelComment"]
            snippet = comment["snippet"]

            comments.append({
                # kolom lama (tidak diubah)
                "text": snippet["textDisplay"],
                "likes": snippet["likeCount"],
                "publishedAt": snippet["publishedAt"],
                # kolom tambahan
                "videoId": video_id,
                "videoUrl": f"https://www.youtube.com/watch?v={video_id}",
                "commentId": comment["id"],
                "authorDisplayName": snippet.get("authorDisplayName"),
                "authorChannelId": snippet.get("authorChannelId", {}).get("value"),
                "updatedAt": snippet.get("updatedAt"),
                "totalReplyCount": thread.get("totalReplyCount"),
                "textOriginal": snippet.get("textOriginal"),
            })

        request = youtube.commentThreads().list_next(request, response)

    return comments


## Running


In [ ]:
all_comments = []
failed = []

for video_id in VIDEO_IDS:
    try:
        result = scrape_comments(video_id)
        all_comments.extend(result)
        print(f"{video_id}: {len(result)} komentar")
    except HttpError as e:
        failed.append(video_id)
        print(f"{video_id}: GAGAL -> {e}")

df = pd.DataFrame(all_comments)

# Buang duplikat kalau notebook dijalankan ulang atau video yang sama muncul dua kali
if "commentId" in df.columns:
    df = df.drop_duplicates(subset=["commentId"], keep="first").reset_index(drop=True)

print(f"\nTotal komentar: {len(df)}")
if failed:
    print(f"Video gagal: {', '.join(failed)}")

df.head()


In [ ]:
print(f"Jumlah data yang berhasil di scrape: {len(df)}")
df.isnull().sum()


## Simpan File


In [ ]:
os.makedirs("../results/youtube", exist_ok=True)

df.to_csv("../results/youtube/youtube_comments_all.csv", index=False, encoding="utf-8-sig")
df.to_excel("../results/youtube/youtube_comments_all.xlsx", index=False)

print(f"Tersimpan: {len(df)} komentar -> youtube_comments_all.csv / .xlsx")
